In [2]:
import pandas as pd
import numpy as np
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Libraries Loaded!")

Libraries Loaded!


In [5]:
response = requests.get("http://localhost:8080/api/players")
df=pd.DataFrame(response.json())

#Create a tier label based on KD Ratio & Win Rate
def assign_tier(row):
    if row['kdRatio'] >= 3.0 and row['winRate']>=0.70:
        return 'Pro'
    elif row['kdRatio'] >= 2.0 and row['winRate']<=0.60:
        return 'Gold'
    elif row['kdRatio'] >= 1.5 and row['winRate']<=0.50:
        return 'Silver'
    else :
        return 'Bronze'

df['tier']= df.apply(assign_tier, axis=1)
print(df[['name','kdRatio','winRate','tier']])

       name  kdRatio  winRate    tier
0  Trinanda      3.5     0.78     Pro
1      Adam      2.5     0.65  Bronze
2    Ricky       1.8     0.52  Bronze
3    Pandda      3.2     0.70     Pro
4       Sid      1.2     0.48  Bronze
5        DJ      3.1     0.71     Pro


In [7]:
# Feature engineering
x=df[['kdRatio','winRate']]
y= df['tier']

#Split Data
x_train, x_test, y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)
#Train Random Forest model
model= RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(x_train,y_train)

#Evaluate
y_pred= model.predict(x_test)
print(f"Accuracy: {accuracy_score(y_test,y_pred):.2%}")
print("\n Classification Report")
print(classification_report(y_test,y_pred))


Accuracy: 50.00%

 Classification Report
              precision    recall  f1-score   support

      Bronze       0.00      0.00      0.00         1
         Pro       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



C:\Users\amlan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\amlan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\amlan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [8]:
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/player_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [9]:
#Test Prediction
loaded_model= joblib.load('../models/player_model.pkl')

#Predict tier for a new player
new_player = pd.DataFrame({
    'kdRatio': [3.5],
    'winRate':[0.78],
})

prediction= loaded_model.predict(new_player)
print(f"Predicted tier: {prediction[0]}")


Predicted tier: Pro
